# Module 1 — What "System 2" Buys You

*Reasoning & System 2: from classical methods to language models*

---

**You will be able to:**

1. State the difference between an answer that is *recalled* and an answer
   that is *computed*, and design an experiment that tells them apart.
2. Implement the cube-rotation algorithm this whole course keeps returning to.
3. Measure **accuracy as a function of problem depth** — the single most
   informative plot in reasoning research.
4. Record deliberation as a **trace**, and score a trace *step by step*
   rather than only on its final answer.
5. Explain why a benchmark must be checked before it is trusted (and find a
   real leak in this repo's own data while doing it).

**Prerequisites:** comfortable Python (dicts, functions, comprehensions). No
maths beyond arithmetic. No libraries beyond the standard library.

**Time:** ~60 minutes for the lecture, ~90 for the exercises and project.

## 1. Two systems

Kahneman's *Thinking, Fast and Slow* (2011) popularised a distinction
psychologists had been circling for decades:

| | System 1 | System 2 |
|---|---|---|
| speed | immediate | slow |
| effort | none | costly, serial |
| control | automatic | deliberate |
| typical use | reading a word, recognising a face | multiplying 17 × 24 |

The canonical probe is the bat-and-ball problem: *a bat and a ball cost
\$1.10 together; the bat costs \$1.00 more than the ball; how much is the
ball?* Almost everyone's first answer is 10¢. It is wrong (the ball is 5¢);
the wrong answer arrives instantly and feels right, and only a deliberate
check catches it.

The framing is contested as psychology — there is no anatomical "System 1"
— but it is an extremely useful engineering distinction, and it is the one
this course is built on:

> **System 1** = a fixed amount of computation per question. A lookup, a
> pattern match, one forward pass.
>
> **System 2** = an amount of computation that *grows with the difficulty of
> the question*. A search, a simulation, a proof, a chain of thought.

That is the operational definition. It is measurable, and it makes an
immediate prediction: a System-1 solver's accuracy must **fall as problems
get deeper**, because the work required grows while the work performed does
not. A System-2 solver's accuracy should stay flat.

The rest of this module turns that prediction into an experiment. The rest
of the *course* builds the classical machinery — logic, search, constraint
propagation, planning, probabilistic inference — that made deliberate
computation precise decades before anyone asked a language model to "think
step by step".

In [ ]:
# --- setup: make `csai` importable no matter where you launched Jupyter ---
import sys
import pathlib

_here = pathlib.Path.cwd()
_course = next(p for p in [_here, *_here.parents] if (p / "csai").is_dir())
if str(_course) not in sys.path:
    sys.path.insert(0, str(_course))

import random
from collections import Counter

from csai import data
from csai.check import checker
from csai.render import bar_chart, table
from csai.trace import Trace, diff_traces

print("course root:", _course)
print("splits found:", data.available_splits())

## 2. The task we will keep coming back to

`color_cube_rotation`, from the [reasoning-gym](https://github.com/open-thought/reasoning-gym)
benchmark suite. A cube has six coloured faces. A sequence of rotations is
applied, each described as *"the cube is rotated so that the side which was
at X is now on top"*. The question asks for the colour of one named face
afterwards.

It is a good teaching task for three reasons:

* The correct algorithm is tiny — six lines — so nothing is hidden.
* Difficulty is a **single integer**: the number of rotations. That gives us
  a clean depth axis to plot accuracy against.
* The repo already ships 11,000 generated problems *with the ground-truth
  state after every intermediate step*, so we can grade reasoning, not just
  answers.

In [ ]:
example = data.load_split("test_seen", limit=1)[0]

print(example["question"])
print("\n--- gold answer ---")
print(example["answer"])
print("\n--- gold reasoning trace (generated by the simulator, not a model) ---")
print(example["cot_trace"])
print("\n--- structured metadata ---")
for k, v in example["metadata"].items():
    print(f"{k:>14}: {v}")

### The rotation rules

A rotation names the face that ends up on top. Four faces move in a cycle
and two stay put (they are the axis of the turn):

| rotate `side` to top | cycle (each face's colour moves to the next) | unchanged |
|---|---|---|
| `front`  | front → top → back → bottom → front | left, right |
| `back`   | back → top → front → bottom → back  | left, right |
| `right`  | right → top → left → bottom → right | front, back |
| `left`   | left → top → right → bottom → left  | front, back |
| `bottom` | top ↔ bottom, and front ↔ back      | left, right |

(`top` is never used as a rotation — it would be the identity.)

`csai.data.simulate` wraps the repo's verified implementation
(`data_gen/cube.py`, ported from reasoning-gym itself). We will use it as
the reference oracle; **Exercise 1 asks you to write your own and check it
against this one.**

In [ ]:
state0 = data.initial_state(example)
rotations = example["metadata"]["rotations"]

print("initial:", state0)
for i, (side, st) in enumerate(zip(rotations, data.simulate(state0, rotations)), 1):
    print(f"after {i} ({side:>6} to top):", st)

target = example["metadata"]["target_side"]
print(f"\nanswer = state[{target!r}] =",
      data.simulate(state0, rotations)[-1][target])

## 3. Four solvers, one experiment

Now the experiment. Four solvers, ordered from pure System 1 to pure
System 2. Each takes an example and returns a colour string.

1. **`constant_solver`** — always say `"yellow"`. Pure prior, zero
   computation. This is the floor.
2. **`random_face_solver`** — pick uniformly among the six colours actually
   on this cube. A smarter prior: it reads the question, but does not
   process it.
3. **`shortcut_solver`** — answer with the colour that started on the target
   face, ignoring the rotations entirely. This is the interesting one: it is
   exactly the kind of plausible surface heuristic a pattern matcher
   latches onto, and it is *right surprisingly often* on short problems.
4. **`lookup_solver`** — memorise every training problem and look the
   question up verbatim. The purest System 1 there is: infinite storage,
   zero computation.
5. **`simulation_solver`** — actually turn the cube. System 2.

In [ ]:
SAMPLE = 400  # keep the notebook fast; raise it if you want tighter estimates

seen = data.load_split("test_seen", limit=SAMPLE)
extrapolate = data.load_split("test_extrapolate", limit=SAMPLE)
train = data.load_split("train", limit=8000)

_rng = random.Random(0)


def constant_solver(ex):
    return "yellow"


def random_face_solver(ex):
    return _rng.choice(list(data.initial_state(ex).values()))


def shortcut_solver(ex):
    """Ignore the rotations; answer with whatever started on the target face."""
    return data.initial_state(ex)[ex["metadata"]["target_side"]]


_memory = {ex["question"]: ex["answer"] for ex in train}


def lookup_solver(ex):
    return _memory.get(ex["question"], "yellow")


def simulation_solver(ex):
    """System 2: do the work. One rotation applied per rotation described."""
    states = data.simulate(data.initial_state(ex), ex["metadata"]["rotations"])
    return states[-1][ex["metadata"]["target_side"]]


def accuracy(solver, examples):
    return sum(solver(ex) == ex["answer"] for ex in examples) / len(examples)


SOLVERS = [
    ("constant", constant_solver),
    ("random face", random_face_solver),
    ("shortcut", shortcut_solver),
    ("lookup", lookup_solver),
    ("simulation", simulation_solver),
]

rows = [(name, f"{accuracy(f, seen):.0%}", f"{accuracy(f, extrapolate):.0%}")
        for name, f in SOLVERS]
print(table(rows, ["solver", "test_seen (1-3 rot)", "test_extrapolate (4-6 rot)"],
            align="lrr"))

### Accuracy against depth

The aggregate table already hints at the story, but the depth curve is what
actually makes the argument. Difficulty here is one number, so we can plot
accuracy against it directly.

In [ ]:
def accuracy_by_depth(solver, examples):
    """Accuracy bucketed by number of rotations."""
    buckets = data.group_by_length(examples)
    return {k: accuracy(solver, group) for k, group in buckets.items()}


all_examples = seen + extrapolate
for name, f in SOLVERS:
    curve = accuracy_by_depth(f, all_examples)
    print(bar_chart(curve.items(), width=34, maximum=1.0,
                    title=f"{name}: accuracy by number of rotations",
                    value_fmt="{:.0%}"))
    print()

Read those curves carefully, because everything else in this course is a
variation on them.

* **`shortcut`** is the honest System-1 story: about 33% at one rotation,
  then down to a plateau in the low twenties. Well above the ~16% from
  guessing a visible face, and nowhere near solving anything. And the 33% is
  not luck — every rotation leaves exactly **two of the six faces fixed**
  (the axis it turns about), so on a one-rotation problem there is a 2-in-6
  chance the question happens to ask about a face that never moved. The
  shortcut has captured a genuine regularity of the task; it simply cannot
  compose. If you only ever tested at depth 1 you would call it promising.
* **`lookup`** is a cliff, not a slope. Perfect where it has seen the
  problem, and the moment it hasn't it falls all the way back to the
  constant baseline. Memorisation does not degrade gracefully; it stops.
* **`simulation`** is flat at 100%, including on chain lengths its author
  never looked at. That flatness *is* the signature of having an algorithm.

This is why `data/test_extrapolate.jsonl` exists in this repo, and why
`PLAN.md` makes generalisation-beyond-trained-depth its third research
question. Aggregate accuracy on problems drawn from the training
distribution cannot distinguish a solver that computes from one that
recalls. **The depth curve can.**

## 4. Check the benchmark before you trust it

The `lookup` row above should have bothered you. It scored 100% on
`test_seen` — a *test* split. Memorising the training set is supposed to
tell you nothing about a held-out set.

The first act of System 2 is to check the premises. Let us check this one.

In [ ]:
train_questions = {ex["question"] for ex in train}
for split_name in ("val", "test_seen", "test_extrapolate"):
    split = data.load_split(split_name, limit=1000)
    leaked = sum(ex["question"] in train_questions for ex in split)
    verdict = "CLEAN" if leaked == 0 else f"LEAKED {leaked / len(split):.0%}"
    print(f"{split_name:>18}: {leaked:>4}/{len(split)} questions also in train   {verdict}")

If your run reports a leak, here is the cause. `data_gen/generate_dataset.py`
builds example *i* of a split from `random.Random(seed + i)`, with these
split definitions:

```python
("train",            1000, 8000, 1, 3),   # uses seeds 1000 .. 8999
("val",              2000, 1000, 1, 3),   # uses seeds 2000 .. 2999  ⊂ train
("test_seen",        3000, 1000, 1, 3),   # uses seeds 3000 .. 3999  ⊂ train
("test_extrapolate", 4000, 1000, 4, 6),   # 4000 .. 4999, but a different
                                          # rotation range, so genuinely new
```

The seed *ranges overlap*. `val` and `test_seen` are not disjoint samples —
they are exact copies of slices of the training set. Any model trained on
`train` will score near-perfectly on them regardless of whether it learned
anything, and the study's "seen" numbers would be uninterpretable.

The fix is one line — spread the base seeds far apart (`1_000_000`,
`2_000_000`, `3_000_000`, `4_000_000`) and regenerate — but notice what
actually caught it: not a test suite, but **a baseline that scored
suspiciously well**. An implausibly strong result is evidence about your
data before it is evidence about your method.

`test_extrapolate` is unaffected: its rotation range (4–6) appears in no
training example, so it is genuinely held out. That is why the capstone in
Module 12 leans on it.

## 5. Traces: grading the reasoning, not just the answer

A final answer is one token of evidence about a process that took several
steps. If a solver is right, was it right *for the right reasons*? If it is
wrong, *where* did it go wrong?

A **trace** answers both. `csai.trace.Trace` is a list of `Step`s, each
pairing an action with the state it produced. Every method in this course
produces one: DPLL's decision stack, A*'s expanded path, a Prolog proof, a
STRIPS plan — and a language model's chain of thought.

In [ ]:
def traced_simulation(ex):
    """Solve, and record what happened on the way."""
    state = data.initial_state(ex)
    tr = Trace(name="cube rotation", initial=state)
    for side in ex["metadata"]["rotations"]:
        state = data.simulate(state, [side])[-1]
        tr.step(f"rotate {side} to top", state)
    return tr.finish(state[ex["metadata"]["target_side"]])


deep = [ex for ex in extrapolate if data.num_rotations(ex) >= 5][0]
print(traced_simulation(deep).render())

Now a solver with a *bug*, not noise: it treats a `bottom` rotation as if it
only swapped top and bottom, forgetting that front and back swap too. Such a
solver still produces a fluent, confident, complete trace.

In [ ]:
def buggy_rotate(state, side):
    s = dict(state)
    if side == "bottom":
        s["top"], s["bottom"] = state["bottom"], state["top"]   # bug: front/back
        return s                                                # should swap too
    return data.simulate(state, [side])[-1]


def traced_buggy(ex):
    state = data.initial_state(ex)
    tr = Trace(name="buggy", initial=state)
    for side in ex["metadata"]["rotations"]:
        state = buggy_rotate(state, side)
        tr.step(f"rotate {side} to top", state)
    return tr.finish(state[ex["metadata"]["target_side"]])


right_answer_wrong_reasons = 0
for ex in extrapolate:
    pred = traced_buggy(ex)
    d = diff_traces(pred, data.gold_states(ex))
    if pred.result == ex["answer"] and not d.identical:
        right_answer_wrong_reasons += 1

print(f"buggy solver, final-answer accuracy : "
      f"{accuracy(lambda e: traced_buggy(e).result, extrapolate):.0%}")
print(f"of {len(extrapolate)} problems, {right_answer_wrong_reasons} were answered "
      f"correctly\ndespite a trace that had already gone wrong.\n")

case = next(ex for ex in extrapolate
            if traced_buggy(ex).result == ex["answer"]
            and not diff_traces(traced_buggy(ex), data.gold_states(ex)).identical)
print("an example of exactly that:\n")
print(diff_traces(traced_buggy(case), data.gold_states(case)))

Right answer, broken reasoning. Grade only the final answer and this solver
looks partly competent; grade the trace and the bug is localised to a
specific step, and the fix is obvious.

Two numbers do that work, and they recur through the whole course:

* **step accuracy** — what fraction of the gold intermediate states did the
  solver reproduce?
* **first divergence** — the index of the earliest step that went wrong.
  Its distribution over a dataset tells you whether a solver fails early
  (doesn't understand the setup) or late (loses track under depth).

`PLAN.md` §5 calls this "step-level faithfulness" and makes it a primary
metric of the study this repo exists to run. You are about to build it.

## 6. Bridge to language models

Nothing above involved a neural network, and every idea in it is now load
bearing in LLM reasoning research:

| what you just did | its modern name |
|---|---|
| made the intermediate states explicit before answering | **chain-of-thought prompting** (Wei et al., 2022) |
| wrote the intermediate states into the training target | **scratchpad supervision** (Nye et al., 2021) — this repo's `messages_cot` |
| compared against a variant trained on answers only | this repo's `messages_answer_only` ablation |
| tested on chain lengths never trained on | **length generalisation**, this repo's `test_extrapolate` |
| scored each intermediate step against a simulator | **process supervision** (Lightman et al., 2023) |
| noticed a right answer reached by broken reasoning | **unfaithful chain-of-thought** (Turpin et al., 2023) |

A language model asked to answer immediately is doing System 1: fixed
computation, no matter the depth. Asked to reason step by step, it spends
computation proportional to the number of steps it writes — the tokens *are*
the working memory. That is why chain of thought helps on multi-step
problems and does nothing for one-step recall.

What classical AI contributes, and what the following eleven modules teach,
is the part that is *not* automatic: how to represent a state, how to search
a space of possibilities, how to propagate a constraint, how to check an
answer cheaply. Modern systems are rediscovering all of it — tree-of-thought
is a search algorithm, self-consistency is Monte-Carlo marginalisation,
tool use is delegation to an interpreter. Learn the classical version once
and you will recognise every one of them.

---
## Exercises

Each stub returns `None`; fill it in and re-run the `check_...()` cell below
it. Checks never raise — a failing check just prints what it wanted. Reference
solutions are in `course/solutions/m01.py`; try each one honestly first.

### Exercise 1 — implement the rotation

Write `rotate(state, side)`: given a `{face: colour}` dict and the face to
bring to the top, return a **new** dict for the resulting cube. Use the cycle
table from Section 2. Do not call `data.simulate`.

<details><summary>Hint</summary>

Each rotation is a 4-cycle of faces. For `front`, the colour on `front` moves
to `top`, `top`'s moves to `back`, `back`'s to `bottom`, `bottom`'s to
`front`; `left` and `right` are untouched. Writing the cycles as a dict of
tuples, `{"front": ("front", "top", "back", "bottom"), ...}`, lets one loop
handle all of them. `bottom` is a double swap rather than a 4-cycle — or,
equivalently, the 4-cycle `("bottom", "top", "front", "back")` is *not* it;
check your answer against `data.simulate`.
</details>

In [ ]:
def rotate(state, side):
    """Return the cube state after bringing `side` to the top."""
    # TODO: build and return a new {face: colour} dict
    return None

In [ ]:
@checker("Exercise 1 — rotate")
def check_ex1():
    rng = random.Random(7)
    colours = ["red", "green", "blue", "yellow", "white", "orange"]
    for _ in range(20):
        shuffled = colours[:]
        rng.shuffle(shuffled)
        st = dict(zip(data.SIDES, shuffled))
        for side in ("front", "back", "left", "right", "bottom"):
            yield (f"rotate {side} to top",
                   rotate(st, side), data.simulate(st, [side])[-1])
    st = dict(zip(data.SIDES, colours))
    yield "does not mutate its argument", (rotate(st, "front"), st)[1], dict(
        zip(data.SIDES, colours))


check_ex1()

### Exercise 2 — solve by simulation

Write `solve_by_simulation(example)` using **your** `rotate`: apply every
rotation in `example["metadata"]["rotations"]` in order, then return the
colour on `example["metadata"]["target_side"]`.

In [ ]:
def solve_by_simulation(example):
    """Return the answer colour, computed by turning the cube."""
    # TODO: chain your rotate() over the rotation list, then read the target
    return None

In [ ]:
@checker("Exercise 2 — solve_by_simulation")
def check_ex2():
    sample = seen[:50] + extrapolate[:50]
    wrong = [ex["id"] for ex in sample if solve_by_simulation(ex) != ex["answer"]]
    yield "correct on 100 problems of every depth", len(wrong), 0
    yield ("independent of depth (works on 6-rotation problems)",
           solve_by_simulation(deep), deep["answer"])


check_ex2()

### Exercise 3 — a second System-1 shortcut

`shortcut_solver` ignored the rotations. Here is a different plausible
shortcut: **apply only the last rotation** and ignore all the earlier ones —
a solver with a working memory of exactly one step.

Write `last_rotation_only(example)`, then run the cell after it to compare
its depth curve with the other baselines. Before you run it: at one
rotation, will it be right always, sometimes, or never?

In [ ]:
def last_rotation_only(example):
    """Apply only the final rotation, starting from the initial state."""
    # TODO: one rotation, from the initial state, then read the target side
    return None

In [ ]:
@checker("Exercise 3 — last_rotation_only")
def check_ex3():
    one = [ex for ex in seen if data.num_rotations(ex) == 1][:20]
    yield ("exact on 1-rotation problems",
           all(last_rotation_only(ex) == ex["answer"] for ex in one), True)
    deeper = [ex for ex in extrapolate if data.num_rotations(ex) >= 5][:60]
    acc = sum(last_rotation_only(ex) == ex["answer"] for ex in deeper) / len(deeper)
    yield (f"but not on deep ones (measured {acc:.0%}, expected below 60%)",
           acc < 0.6, True)


check_ex3()

In [ ]:
# Compare the two shortcuts against depth (run after Exercise 3 passes).
if last_rotation_only(seen[0]) is not None:
    for name, f in [("shortcut (ignore all)", shortcut_solver),
                    ("last rotation only", last_rotation_only)]:
        print(bar_chart(accuracy_by_depth(f, all_examples).items(),
                        width=34, maximum=1.0, value_fmt="{:.0%}",
                        title=f"{name}: accuracy by depth"))
        print()

### Exercise 4 — accuracy by depth, from scratch

Write `accuracy_by_length(solver, examples)` returning
`{num_rotations: accuracy}`, sorted by key. Do not call
`data.group_by_length` or the lecture's `accuracy_by_depth` — the point is
to write the measurement yourself, since you will lean on it all course.

In [ ]:
def accuracy_by_length(solver, examples):
    """Return {number of rotations: fraction correct}, keys ascending."""
    # TODO: bucket by data.num_rotations(ex), then score each bucket
    return None

In [ ]:
@checker("Exercise 4 — accuracy_by_length")
def check_ex4():
    got = accuracy_by_length(simulation_solver, all_examples)
    yield "returns a dict", isinstance(got, dict), True
    yield "one entry per depth present", sorted(got or {}), [1, 2, 3, 4, 5, 6]
    yield "keys ascending", list(got or {}), sorted(got or {})
    yield "a correct solver scores 1.0 everywhere", set((got or {}).values()), {1.0}
    yield ("agrees with the lecture version on a shortcut solver",
           accuracy_by_length(shortcut_solver, all_examples),
           accuracy_by_depth(shortcut_solver, all_examples))


check_ex4()

### Exercise 5 — produce a trace

Write `trace_for(example)` returning a `csai.trace.Trace` whose steps are the
cube states after each rotation, whose `initial` is the starting state, and
whose `result` is the final answer. Use `Trace(...)`, `.step(action, state)`
and `.finish(answer)`.

In [ ]:
def trace_for(example):
    """Return a Trace of the full simulation, finished with the answer."""
    # TODO: Trace(name=..., initial=...), one .step() per rotation, .finish()
    return None

In [ ]:
@checker("Exercise 5 — trace_for")
def check_ex5():
    tr = trace_for(deep)
    yield "returns a Trace", isinstance(tr, Trace), True
    yield "one step per rotation", len(tr or []), data.num_rotations(deep)
    yield "states match the simulator", (tr.states if tr else None), data.gold_states(deep)
    yield "initial state recorded", (tr.initial if tr else None), data.initial_state(deep)
    yield "finished with the answer", (tr.result if tr else None), deep["answer"]
    short = trace_for(next(e for e in seen if data.num_rotations(e) == 1))
    yield "works at depth 1 too", len(short or []), 1


check_ex5()

### Exercise 6 — first divergence

Write `first_divergence(predicted, gold)` taking two lists of states and
returning the **1-based index** of the first position where they differ, or
`None` if the predicted list reproduces every gold state. If `predicted` is
shorter than `gold`, the first missing position counts as a divergence.

This is `csai.trace.diff_traces` reimplemented; write it before you use it.

In [ ]:
def first_divergence(predicted, gold):
    """1-based index of the first differing state, or None if all match."""
    # TODO: compare position by position; a missing prediction counts as wrong
    return None

In [ ]:
@checker("Exercise 6 — first_divergence")
def check_ex6():
    yield "identical lists", first_divergence([1, 2, 3], [1, 2, 3]), None
    yield "differs at the start", first_divergence([9, 2, 3], [1, 2, 3]), 1
    yield "differs at the end", first_divergence([1, 2, 9], [1, 2, 3]), 3
    yield "predicted too short", first_divergence([1], [1, 2, 3]), 2
    yield "both empty", first_divergence([], []), None
    yield "predicted too long is fine if the prefix matches", first_divergence(
        [1, 2, 3, 4], [1, 2, 3]), None
    pred = traced_buggy(case).states
    yield ("agrees with csai.trace on the buggy solver",
           first_divergence(pred, data.gold_states(case)),
           diff_traces(pred, data.gold_states(case)).first_divergence)


check_ex6()

---
## Project — a trace-aware evaluation harness

Build the measurement instrument the rest of the course will use.

Write `evaluate(traced_solver, examples)` where `traced_solver(example)`
returns a `Trace`. Return a dict with exactly these keys:

| key | value |
|---|---|
| `"n"` | number of examples evaluated |
| `"accuracy"` | fraction whose `trace.result` equals the gold answer |
| `"by_length"` | `{num_rotations: accuracy}`, keys ascending |
| `"step_accuracy"` | mean over examples of (gold steps reproduced ÷ gold steps) |
| `"first_wrong"` | `{step index: count}`, using **key `0` for traces that never diverged** |
| `"lucky"` | how many examples had the right answer *and* a diverged trace |

`"lucky"` is the point of the whole exercise: it counts answers that were
right for the wrong reasons, which final-answer accuracy cannot see.

Then use the provided `noisy_solver(p, seed)` — which corrupts each step's
state with probability `p` and carries the corruption forward, like a model
losing track mid-chain — to answer two questions in the write-up cell:

1. How does `step_accuracy` compare to `accuracy` as `p` rises? Which one
   degrades more smoothly, and why is that the more useful signal?
2. Where does the `first_wrong` distribution concentrate, and what does its
   shape tell you about *how* a noisy reasoner fails on deep problems?

In [ ]:
def noisy_solver(p, seed=0):
    """A solver that simulates correctly but corrupts each step with prob. p.

    Once a state is corrupted the error is carried forward, exactly as a
    reasoner losing track mid-chain would: later steps are computed from the
    wrong state, not the right one.
    """
    def solve(example):
        rng = random.Random(f"{seed}:{example['id']}")
        state = data.initial_state(example)
        tr = Trace(name=f"noisy p={p}", initial=state)
        for side in example["metadata"]["rotations"]:
            state = data.simulate(state, [side])[-1]
            if rng.random() < p:
                faces = list(state)
                a, b = rng.sample(faces, 2)
                state = dict(state)
                state[a], state[b] = state[b], state[a]
            tr.step(f"rotate {side} to top", state)
        return tr.finish(state[example["metadata"]["target_side"]])

    return solve


def evaluate(traced_solver, examples):
    """Return the report described above."""
    # TODO: run the solver on every example, comparing trace.states against
    # data.gold_states(ex); accumulate the six keys.
    return None

In [ ]:
@checker("Project — evaluate")
def check_project():
    sample = (seen[:60] + extrapolate[:60])
    perfect = evaluate(traced_simulation, sample)
    yield "returns a dict", isinstance(perfect, dict), True
    yield "has exactly the required keys", sorted(perfect or {}), [
        "accuracy", "by_length", "first_wrong", "lucky", "n", "step_accuracy"]
    yield "n counts the examples", (perfect or {}).get("n"), len(sample)
    yield "a correct solver is 100% accurate", (perfect or {}).get("accuracy"), 1.0
    yield "…with 100% step accuracy", (perfect or {}).get("step_accuracy"), 1.0
    yield ("…and never diverges (all counted under key 0)",
           (perfect or {}).get("first_wrong"), {0: len(sample)})
    yield "…and is never merely lucky", (perfect or {}).get("lucky"), 0
    yield ("by_length covers every depth",
           sorted((perfect or {}).get("by_length") or {}), [1, 2, 3, 4, 5, 6])

    broken = evaluate(noisy_solver(1.0, seed=1), sample)
    yield ("a solver corrupted at every step always diverges at step 1",
           (broken or {}).get("first_wrong"), {1: len(sample)})
    yield ("…yet still gets some answers right by luck",
           0 < (broken or {}).get("lucky", 0), True)

    partial = evaluate(noisy_solver(0.3, seed=2), sample)
    yield ("partial noise gives partial step accuracy",
           0.0 < (partial or {}).get("step_accuracy", 0.0) < 1.0, True)
    yield ("step accuracy is at least final-answer accuracy here",
           (partial or {}).get("step_accuracy", 0) >= (partial or {}).get("accuracy", 1) - 0.35,
           True)


check_project()

In [ ]:
# Run this once `evaluate` works: the noise sweep your write-up discusses.
if evaluate(traced_simulation, seen[:5]) is not None:
    rows = []
    for p in (0.0, 0.1, 0.2, 0.4, 0.8, 1.0):
        r = evaluate(noisy_solver(p, seed=3), all_examples)
        div = {k: v for k, v in sorted(r["first_wrong"].items()) if k}
        common = max(div, key=div.get) if div else "-"
        rows.append((f"{p:.1f}", f"{r['accuracy']:.0%}", f"{r['step_accuracy']:.0%}",
                     r["lucky"], common))
    print(table(rows, ["noise p", "answer acc", "step acc", "lucky", "modal first-wrong step"],
                align="rrrrr"))

### Write-up

Replace this cell with a few sentences answering the project's two
questions. A good answer names *which metric you would monitor during
training and why*, given that both are available to you.

---
## Further reading

* D. Kahneman, *Thinking, Fast and Slow* (2011) — chapters 1–3 for the
  framing; note the replication debates around the priming literature.
* J. Evans & K. Stanovich, "Dual-Process Theories of Higher Cognition:
  Advancing the Debate" (2013) — the careful version of the distinction.
* A. Newell & H. Simon, *Human Problem Solving* (1972) — where "problem
  space search" as a model of deliberate thought comes from. Module 6 is
  downstream of this book.
* M. Nye et al., "Show Your Work: Scratchpads for Intermediate Computation
  with Language Models" (2021) — the direct ancestor of this repo's
  `messages_cot` split.
* J. Wei et al., "Chain-of-Thought Prompting Elicits Reasoning in Large
  Language Models" (2022).
* M. Turpin et al., "Language Models Don't Always Say What They Think"
  (2023) — unfaithful reasoning, i.e. the buggy solver of Section 5.

**Next:** Module 2 asks what it even means for a conclusion to *follow* from
what you know — propositional logic, models, and entailment.